# Validation on confirmed TESS planets

This notebook runs the `tess-transit-hunter` pipeline on confirmed planets and compares the
recovered period, depth and radius with the NASA Exoplanet Archive. It is the interactive
companion of `scripts/validate_known_planets.py`; both use the functions in
`transit_hunter.validation`.

**Requirements:** network access to `exoplanetarchive.ipac.caltech.edu` (reference values) and
`mast.stsci.edu` (light curves and the TIC). Light curves are cached, so re-running is fast.

The cell outputs are intentionally not stored in the repository. Run the notebook (or the
script) to produce them; the numbers quoted in the README and docs come from
`results/validation/validation.json`, which the script writes.

In [ ]:
import logging
from dataclasses import replace
from pathlib import Path

from IPython.display import Image, Markdown, display

from transit_hunter.catalog import get_stellar_params, query_confirmed_planets
from transit_hunter.data import fetch_lightcurve
from transit_hunter.fit import FitConfig
from transit_hunter.pipeline import PipelineConfig, run_on_lightcurve
from transit_hunter.search import default_n_workers
from transit_hunter.validation import DEFAULT_TARGETS, compare_planet, comparison_markdown

logging.basicConfig(level=logging.INFO)
OUT = Path("../results/validation")
WORKERS = default_n_workers()
QUICK = True  # short MCMC chains for interactive use; set False for publication-quality fits

## 1. Reference values from the NASA Exoplanet Archive

`pscomppars` has one row per planet (parameters can come from different papers; see the
archive documentation). Only planets flagged as transiting are kept.

In [ ]:
# Planets are matched by TIC ID: the archive lists pi Men as HD 39091 and HD 21749 as GJ 143.
published = query_confirmed_planets([t.tic_id for t in DEFAULT_TARGETS])
for p in sorted(published, key=lambda p: (p.host, p.period or 0)):
    print(f"{p.name:14s} TIC {p.tic_id}  P = {p.period} d  Rp/R* = {p.rp_rs}  Rp = {p.rp_earth} R_earth")

## 2. Run the pipeline on one host

Pick a host from the list above. The report folder contains every figure and a `report.json`.

In [ ]:
target = next(t for t in DEFAULT_TARGETS if t.host == "WASP-18")
host, tic = target.host, target.tic_id
planets = [p for p in published if p.tic_id == tic]

fit_cfg = FitConfig(n_workers=WORKERS)
if QUICK:
    fit_cfg = replace(fit_cfg, n_walkers=32, max_steps=3000, min_steps=1000)
config = PipelineConfig()
config = replace(config, search=replace(config.search, n_workers=WORKERS), fit=fit_cfg)

lc = fetch_lightcurve(tic)
stellar = get_stellar_params(tic, lc.meta.get("stellar_header"))
print(f"TIC {tic}: sectors {lc.sectors}, {len(lc)} points; stellar parameters from {stellar.source}")
report = run_on_lightcurve(lc, OUT / host.replace(" ", "_"), stellar, config, name=host)

In [ ]:
folder = OUT / host.replace(" ", "_")
display(Image(filename=str(folder / "search_summary.png")))
for i, _ in enumerate(report["planets"], 1):
    display(Image(filename=str(folder / f"fit_{i}.png")))
    display(Image(filename=str(folder / f"vetting_{i}.png")))

## 3. Recovered vs published values

In [ ]:
rows = [compare_planet(p, report) for p in planets]
display(Markdown(comparison_markdown(rows)))

## 4. Full validation set

`scripts/validate_known_planets.py` loops over every host in `DEFAULT_TARGETS` and writes
`results/validation/validation.md`. After running it, the combined table can be shown here:

In [ ]:
table = OUT / "validation.md"
if table.exists():
    display(Markdown(table.read_text()))
    display(Image(filename=str(OUT / "validation_errors.png")))
else:
    print("Run `python scripts/validate_known_planets.py` first.")